In [ ]:
import os
from circuit_tracer.subgraph.prune import prune_graph_pipeline
from circuit_tracer.subgraph.api import generate_graph, get_feature, save_subgraph
from circuit_tracer.subgraph.classify import classify_features
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils.create_graph_files import create_graph_files  
from circuit_tracer.graph import Graph, prune_graph, compute_graph_scores
from transformers import AutoModelForCausalLM, AutoTokenizer
import shap

True

## Load LLM

In [2]:
model_name = "google/gemma-2-2b" # google/gemma-scope-2-4b-it crosscoder/layer_9_17_22_29_width_65k_l0_medium
# transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
# backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
# model = ReplacementModel.from_pretrained(
#     model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
# )

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": False, # set to False for deterministic output
    "max_new_tokens": 1, # set to 1 for single-token generation
    "temperature": 0, # set to 0 for deterministic output
    # "no_repeat_ngram_siz e": 2, 
}

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Use SHAP for token weights

In [ ]:
explainer = shap.Explainer(model, tokenizer)

In [ ]:
prompts = ['Fact: The capital of the state containing Dallas is']

In [ ]:
shap_values = explainer(prompts)

In [ ]:
print(shap_values.values.squeeze())

[-3.50681455e-06  2.76400235e+00 -8.72354661e-01  2.00342429e+00
  4.72185472e+00  1.18334559e+00  1.11659276e-01  2.30677287e+00
  9.67523081e-01  5.83311160e+00  2.01544604e+00]


In [ ]:
from scipy.special import softmax
token_weights = softmax(shap_values.values.squeeze())
print(token_weights)

[0.00198786 0.03153391 0.00083086 0.01473883 0.22338926 0.00649094
 0.00222269 0.01996207 0.0052309  0.67869559 0.01491708]


## Generate graph if not exist

In [ ]:
# For now just do it on the web UI since we don't have a dataset yet

## ASG Pipeline

### Prune

In [28]:

json_path = "temp_graph_files/austin.json"
source_set = 'clt-hp' # gemmascope-transcoder-16k
token_weights = [0, 0, 0, 0, 1/3, 0, 0, 1/3, 0, 1/3, 0]
kept_ids, pruned_adj, node_inf, node_rel, attr, metadata = prune_graph_pipeline(
    json_path=json_path,
    logit_weights='target',
    token_weights=token_weights,
    node_influence_threshold=0.6,
    edge_influence_threshold=0.95,
    node_relevance_threshold=1,
    edge_relevance_threshold=1,
    keep_all_tokens_and_logits=True,
)

In [29]:
print(len(kept_ids))

32


In [30]:
for i, node_id in enumerate(kept_ids):
    print(node_id, attr[node_id].get("clerp", ""), node_inf[i].item(), node_rel[i].item())

0_1861_9 Texas 0.6002651453018188 0.8500428199768066
1_12928_10  AssemblyCulture 0.5457153916358948 0.8204886317253113
1_72774_10 Code/licensing snippets 0.5339171290397644 0.8239325881004333
16_89970_9 Texas 0.5723963379859924 0.6424320340156555
18_3623_10 capital cities 0.5764042735099792 0.6277174353599548
20_44686_9 Texas locations/legal contexts 0.5682745575904846 0.5120062232017517
20_44686_10 Texas locations/legal contexts 0.5273332595825195 0.5191220045089722
20_74108_10 capital 0.5873730778694153 0.5618082284927368
21_84338_10 Cities and locations 0.5839060544967651 0.5022703409194946
22_11998_10 Texas and Dallas 0.5599021315574646 0.494268000125885
22_32893_10 Texas legal documents 0.5907537937164307 0.522559642791748
E_2_0 Emb: " <bos>" 0.3491837978363037 1.0000001192092896
E_18143_1 Emb: " Fact" 0.4144315719604492 1.0000001192092896
E_235292_2 Emb: " :" 0.46905380487442017 1.0000001192092896
E_714_3 Emb: "  The" 0.4942685067653656 1.0000001192092896
E_6037_4 Emb: "  capital

### Classify features

Heuristic classify alg is terrible, needs improvement or just use LLM

In [ ]:
# feature_types = classify_features(kept_ids, attr, metadata)
# print(feature_types)

IndexError: list index out of range

In [ ]:
attr

{'8_37691_9': {'node_id': '8_37691_9',
  'feature': 710663841,
  'layer': '8',
  'ctx_idx': 9,
  'feature_type': 'cross layer transcoder',
  'token_prob': 0,
  'is_target_logit': False,
  'run_idx': 0,
  'reverse_ctx_idx': 0,
  'jsNodeId': '8_37691-0',
  'clerp': 'technical content',
  'influence': 0.6903811097145081,
  'activation': 17.875},
 '10_75793_10': {'node_id': '10_75793_10',
  'feature': 2873161099,
  'layer': '10',
  'ctx_idx': 10,
  'feature_type': 'cross layer transcoder',
  'token_prob': 0,
  'is_target_logit': False,
  'run_idx': 0,
  'reverse_ctx_idx': 0,
  'jsNodeId': '10_75793-0',
  'clerp': ' blues',
  'influence': 0.5932801365852356,
  'activation': 15.5625},
 '16_22056_10': {'node_id': '16_22056_10',
  'feature': 243619684,
  'layer': '16',
  'ctx_idx': 10,
  'feature_type': 'cross layer transcoder',
  'token_prob': 0,
  'is_target_logit': False,
  'run_idx': 0,
  'reverse_ctx_idx': 0,
  'jsNodeId': '16_22056-0',
  'clerp': 'uncommon words',
  'influence': 0.662665